# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Swaraj0204/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:

# Baseline rule:
# Prioritize content that has not been updated for at least 180 days
# and received at least 500 impressions in the last 90 days.
# Rank eligible content by impressions_90d, highest first.

rule_summary = {
    "stale_days_threshold": 180,
    "minimum_impressions_90d": 500,
    "score": "impressions_90d",
    "reason_code": "stale_but_visible",
    "action": "REVIEW_REFRESH"
}

print(rule_summary)


{'stale_days_threshold': 180, 'minimum_impressions_90d': 500, 'score': 'impressions_90d', 'reason_code': 'stale_but_visible', 'action': 'REVIEW_REFRESH'}


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:

!git clone https://github.com/Swaraj0204/flyrank-ml-internship.git /content/flyrank-ml-internship


Cloning into '/content/flyrank-ml-internship'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 148 (delta 58), reused 103 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (148/148), 1.86 MiB | 9.56 MiB/s, done.
Resolving deltas: 100% (58/58), done.


In [4]:

from pathlib import Path
import json
import pandas as pd

# Find the cloned repository in this Colab runtime
ROOT = Path("/content/flyrank-ml-internship")

if not ROOT.exists():
    raise FileNotFoundError(
        "Repository not found at /content/flyrank-ml-internship. "
        "Tell me and we will locate it."
    )

DATA = ROOT / "data/raw/content_refresh_anonymized.csv"
OUT = ROOT / "work/outputs"
OUT.mkdir(parents=True, exist_ok=True)

# Load the dataset
df = pd.read_csv(DATA)

# Baseline rule: stale and still visible in search
stale = df["days_since_last_update"] >= 180
visible = df["impressions_90d"] >= 500
eligible = stale & visible

# Keep only the fields needed for the review queue
ranked = df.loc[eligible, [
    "content_id",
    "client_id",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position"
]].copy()

# Exactly one score, one reason code, and one action
ranked["score"] = ranked["impressions_90d"]
ranked["reason_code"] = "stale_but_visible"
ranked["action"] = "REVIEW_REFRESH"

# Highest score first
ranked = ranked.sort_values(
    "score", ascending=False
).reset_index(drop=True)

ranked.insert(0, "rank", range(1, len(ranked) + 1))

# Save the ranked queue
csv_path = OUT / "baseline_action_score.csv"
ranked.to_csv(csv_path, index=False)

# Save evaluation metrics
metrics = {
    "dataset_rows": int(len(df)),
    "eligible_pages": int(len(ranked)),
    "eligible_pct": round(100 * len(ranked) / len(df), 2),
    "score_definition": "impressions_90d for eligible pages",
    "reason_code": "stale_but_visible",
    "action": "REVIEW_REFRESH",
    "rule": (
        "days_since_last_update >= 180 AND "
        "impressions_90d >= 500"
    ),
    "uses_trend_pct": False,
    "uses_trend_direction": False
}

metrics_path = OUT / "baseline_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2))

print("Dataset rows:", len(df))
print("Eligible pages:", len(ranked))
print("Queue CSV saved:", csv_path.exists())
print("Metrics JSON saved:", metrics_path.exists())
print("\nTop 10 ranked pages:")
display(ranked.head(10))


Dataset rows: 30000
Eligible pages: 17
Queue CSV saved: True
Metrics JSON saved: True

Top 10 ranked pages:


,rank,content_id,client_id,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,score,reason_code,action
0,1,content_cf56e2e2e282,client_7f2253d7e2,194,61678,94,0.15,19.7,61678,stale_but_visible,REVIEW_REFRESH
1,2,content_7368877ea310,client_7f2253d7e2,194,59472,77,0.13,24.8,59472,stale_but_visible,REVIEW_REFRESH
2,3,content_1bfaa38ff26c,client_7f2253d7e2,194,25715,60,0.23,22.2,25715,stale_but_visible,REVIEW_REFRESH
3,4,content_0a91db491d14,client_7f2253d7e2,193,13299,65,0.49,10.5,13299,stale_but_visible,REVIEW_REFRESH
4,5,content_5feee3994adb,client_7f2253d7e2,194,7812,1,0.01,39.0,7812,stale_but_visible,REVIEW_REFRESH
5,6,content_c2d929d83eaa,client_7f2253d7e2,193,7558,15,0.20,17.9,7558,stale_but_visible,REVIEW_REFRESH
6,7,content_b16bd7307b39,client_7f2253d7e2,194,4590,0,0.00,31.0,4590,stale_but_visible,REVIEW_REFRESH
7,8,content_fe16a55cd13d,client_7f2253d7e2,194,4556,15,0.33,16.4,4556,stale_but_visible,REVIEW_REFRESH
8,9,content_ecb6215e79fd,client_7f2253d7e2,194,4429,17,0.38,25.3,4429,stale_but_visible,REVIEW_REFRESH
9,10,content_928af3e22c80,client_7f2253d7e2,193,1697,2,0.12,15.8,1697,stale_but_visible,REVIEW_REFRESH


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:

# Review the top 10 recommendations.
# Explain why each page was selected and what could make
# the recommendation wrong.

top10_review = ranked.head(10).copy()

top10_review["why_selected"] = top10_review.apply(
    lambda row: (
        f"Not updated for {int(row['days_since_last_update'])} days; "
        f"received {int(row['impressions_90d']):,} impressions in 90 days."
    ),
    axis=1
)

top10_review["what_could_make_it_wrong"] = (
    "High impressions do not prove the content is outdated or needs "
    "a refresh. Check search intent, accuracy, current usefulness, "
    "and whether the page already satisfies users."
)

review_columns = [
    "rank",
    "action",
    "reason_code",
    "why_selected",
    "what_could_make_it_wrong"
]

display(top10_review[review_columns])

print("Top-10 review rows:", len(top10_review))


,rank,action,reason_code,why_selected,what_could_make_it_wrong
0,1,REVIEW_REFRESH,stale_but_visible,"Not updated for 194 days; received 61,678 impr...",High impressions do not prove the content is o...
1,2,REVIEW_REFRESH,stale_but_visible,"Not updated for 194 days; received 59,472 impr...",High impressions do not prove the content is o...
2,3,REVIEW_REFRESH,stale_but_visible,"Not updated for 194 days; received 25,715 impr...",High impressions do not prove the content is o...
3,4,REVIEW_REFRESH,stale_but_visible,"Not updated for 193 days; received 13,299 impr...",High impressions do not prove the content is o...
4,5,REVIEW_REFRESH,stale_but_visible,"Not updated for 194 days; received 7,812 impre...",High impressions do not prove the content is o...
5,6,REVIEW_REFRESH,stale_but_visible,"Not updated for 193 days; received 7,558 impre...",High impressions do not prove the content is o...
6,7,REVIEW_REFRESH,stale_but_visible,"Not updated for 194 days; received 4,590 impre...",High impressions do not prove the content is o...
7,8,REVIEW_REFRESH,stale_but_visible,"Not updated for 194 days; received 4,556 impre...",High impressions do not prove the content is o...
8,9,REVIEW_REFRESH,stale_but_visible,"Not updated for 194 days; received 4,429 impre...",High impressions do not prove the content is o...
9,10,REVIEW_REFRESH,stale_but_visible,"Not updated for 193 days; received 1,697 impre...",High impressions do not prove the content is o...


Top-10 review rows: 10


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [7]:

# Identify potentially weak recommendations.
# These are candidates for manual review, not automatically wrong picks.

weak_picks = ranked[
    (ranked["clicks_90d"] == 0) |
    (ranked["avg_position"] >= 20)
].copy()

weak_picks["weak_pick_reason"] = weak_picks.apply(
    lambda row: (
        "No clicks in the last 90 days"
        if row["clicks_90d"] == 0
        else "Average search position is 20 or worse"
    ),
    axis=1
)

print("Potential weak picks:", len(weak_picks))
display(
    weak_picks[
        [
            "rank",
            "days_since_last_update",
            "impressions_90d",
            "clicks_90d",
            "ctr",
            "avg_position",
            "weak_pick_reason"
        ]
    ]
)

# Leakage checks: do not use label-derived trend fields as features.
forbidden_features = {"trend_pct", "trend_direction"}

queue_columns = set(ranked.columns)
leakage_columns_present = forbidden_features.intersection(queue_columns)

print("\nLeakage columns in queue:", leakage_columns_present)
assert not leakage_columns_present, (
    "Leakage detected: trend columns must not be in the queue."
)

# Verify the queue contract.
assert ranked["score"].notna().all()
assert ranked["reason_code"].nunique() == 1
assert ranked["action"].nunique() == 1
assert ranked["rank"].is_monotonic_increasing
assert ranked["score"].is_monotonic_decreasing

print("Leakage check: PASS")
print("Queue contract checks: PASS")


Potential weak picks: 7


,rank,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,weak_pick_reason
1,2,194,59472,77,0.13,24.8,Average search position is 20 or worse
2,3,194,25715,60,0.23,22.2,Average search position is 20 or worse
4,5,194,7812,1,0.01,39.0,Average search position is 20 or worse
6,7,194,4590,0,0.00,31.0,No clicks in the last 90 days
8,9,194,4429,17,0.38,25.3,Average search position is 20 or worse
11,12,194,1316,2,0.15,21.8,Average search position is 20 or worse
16,17,183,533,0,0.00,48.0,No clicks in the last 90 days



Leakage columns in queue: set()
Leakage check: PASS
Queue contract checks: PASS


In [8]:

# Final self-check for the baseline notebook

checks = {
    "Dataset has 30,000 rows": len(df) == 30000,
    "Exactly one score column": list(ranked.columns).count("score") == 1,
    "Exactly one reason code column": list(ranked.columns).count("reason_code") == 1,
    "Exactly one action column": list(ranked.columns).count("action") == 1,
    "Queue is ranked by descending score": ranked["score"].is_monotonic_decreasing,
    "Queue contains no trend leakage columns": not {
        "trend_pct", "trend_direction"
    }.intersection(ranked.columns),
    "Queue CSV exists": csv_path.exists(),
    "Metrics JSON exists": metrics_path.exists(),
    "Top-10 review has 10 rows": len(top10_review) == min(10, len(ranked)),
    "Weak-pick review was created": "weak_pick_reason" in weak_picks.columns,
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} — {name}")

assert all(checks.values()), "One or more self-checks failed."

print(f"\nAll {len(checks)} self-checks passed.")
print("Eligible pages:", len(ranked))
print("Potential weak picks:", len(weak_picks))


PASS — Dataset has 30,000 rows
PASS — Exactly one score column
PASS — Exactly one reason code column
PASS — Exactly one action column
PASS — Queue is ranked by descending score
PASS — Queue contains no trend leakage columns
PASS — Queue CSV exists
PASS — Metrics JSON exists
PASS — Top-10 review has 10 rows
PASS — Weak-pick review was created

All 10 self-checks passed.
Eligible pages: 17
Potential weak picks: 7


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.